### 1. Setup
We start by loading environment variables from a `.env` file and initializing the Gemini client using the `google-genai` SDK.

In [21]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client()
model_id = "gemini-2.5-flash-lite"

### 2. Helper Functions
These helper functions manage the conversation history and facilitate interactions with the Gemini model.

In [22]:
# Helper functions
from google.genai import types

def add_user_message(messages, message):
    messages.append({"role": "user", "parts": [{"text": message}]})

def add_assistant_message(messages, content):
    messages.append({"role": "model", "parts": content.parts})

def chat(messages, system=None, temperature=1.0, tools=None):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        tools=tools,
    )
    
    response = client.models.generate_content(
        model=model_id,
        contents=messages,
        config=config
    )
    return response


### 3. Google Search Tool
Gemini provides a built-in Google Search tool that allows the model to retrieve real-time information from the web.

In [26]:
# Define the Google Search tool
google_search_tool = types.Tool(
    google_search=types.GoogleSearch()
)
tools = [google_search_tool]

### 4. Executing Search
We send a prompt to Gemini with the Google Search tool enabled. Gemini will autonomously decide when to use the tool to provide an up-to-date answer.

In [27]:
messages = []
prompt = "What's the best exercise for gaining leg muscle?"
add_user_message(messages, prompt)

print(f"User: {prompt}\n")

response = chat(messages, tools=tools)

print("Assistant:")
for part in response.candidates[0].content.parts:
    if part.text:
        print(part.text)

# Show grounding metadata if available
if response.candidates[0].grounding_metadata:
    print("\nGrounding Sources:")
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.web:
            print(f"- {chunk.web.title}: {chunk.web.uri}")

User: What's the best exercise for gaining leg muscle?

Assistant:
For building leg muscle, compound exercises that work multiple muscle groups simultaneously are generally considered the most effective. These movements allow you to lift heavier weights, which is crucial for muscle hypertrophy.

Here are some of the best exercises for gaining leg muscle:

*   **Squats:** Often called the "ultimate leg builder," squats engage your quadriceps, hamstrings, glutes, and calves. Variations include the barbell back squat, front squat, goblet squat, and Bulgarian split squat.
*   **Deadlifts:** Particularly the Romanian deadlift (RDL), these exercises are excellent for targeting the hamstrings and glutes, as well as the entire posterior chain.
*   **Lunges:** These are versatile and can be performed in various ways, such as walking lunges, reverse lunges, and lateral lunges, effectively working the quads, hamstrings, and glutes.
*   **Leg Press:** This machine-based exercise allows for targete